In [ ]:
from src.analyse_equations.analyse_equations import add_all_data_error, print_units_of_one_equation
from src.analyse_equations.analyse_equations import set_pandas_options, create_error_table
from src.analyse_equations.analyse_equations import add_n_fold_error

from src.analyse_equations.add_information_to_equations import add_proposed_equations

from src.analyse_equations.plot_error_per_system import plot_error_per_system
from src.analyse_equations.plot_predictions_of_one_equation import plot_prediction
from src.analyse_equations.utils import save_proposed_equation, get_data_folds, load_proposed_equations

from src.config.config_equations_for_each_dataset import ConfigEquationDiscovery
from src.equation_discovery.evaluate_equation import map_equation_to_syntax_tree

from src.preprocess_data.preprocess_data import prepare_dataset, get_unit_dict
from src.config.config_analyse_equations import ConfigPlotBestEquation
from src.config.config_load_dataset import ConfigLoadData
from src.SyntaxTree.src.syntax_tree.config_syntax_tree import ConfigSyntaxTree
from src.config.config_hyperparameter import ConfigHyperparameter
import logging
from src.analyse_equations.create_constant_table import create_constant_table
from src.analyse_equations.plot_abs_difference_between_equation import abs_difference_between_equation
from src.analyse_equations.plot_histogram_for_features import histogram_for_features
from src.analyse_equations.plot_error_per_system import heatmap_error_per_system, save_system_error_heatmap
from src.analyse_equations.create_constant_table import save_constant_table
from src.analyse_equations.example_evaluation import save_example_evaluation_dict, get_example_evaluation_dict
from pathlib import Path

# Configs

In [ ]:
logger = logging.getLogger(__name__)
set_pandas_options()

parser = ConfigHyperparameter.arguments_parser()
parser = ConfigLoadData.arguments_parser(parser)
parser = ConfigEquationDiscovery.arguments_parser(parser)
parser = ConfigPlotBestEquation.arguments_parser(parser)
parser = ConfigSyntaxTree.arguments_parser(parser)
args, unknown = parser.parse_known_args()
args.save_path = args.ROOT_DIR / (f'results/'
                                  f'{Path(*Path(args.path_to_datasets).parts[1:])}'
                                  f'/{args.exp_name}')
args.save_path.mkdir(parents=True, exist_ok=True)
args.unit_dict = get_unit_dict(args)
args.unit_dict['y'] = args.unit_dict[args.target]
args.unit_dimension = 5
measurement_error_dic = {
    'drop_length': 0.000042,
    'adv': 0.07696902,
    'rec': 0.03298672,
    'avg_vel': 0.0021,
    'width': 0.00005,
    'y_center':0.000003,
    'middle_angle': 0.03141593,
    'x_center': 0.0000042,
    'static_adv': 0.01570796,
    'static_rec': 0.01570796
}
logging.basicConfig(level=logging.INFO)

print(f"results are saved to {args.save_path}")


# Results to load

In [ ]:
proposed_equations = load_proposed_equations(args)
add_proposed_equations(args, proposed_equations)

#files_test, files_train = get_train_test_files(args, proposed_equations)
folds_dict = get_data_folds(args, proposed_equations)
all_files = []
for excel_name, files in folds_dict.items():
    all_files.extend(files)

# Cross Validation

In [ ]:
filtered_dfs_test, filtered_dfs_train, tree = add_n_fold_error(args, folds_dict, measurement_error_dic, proposed_equations)
save_proposed_equation(args, folds_dict, proposed_equations)

# Calculate Error over all data

In [ ]:
all_data_dfs = prepare_dataset(args, all_files)
add_all_data_error(all_data_dfs, args, proposed_equations)


# Example Evaluation

In [ ]:
num_variables = 1

example_evaluation_dict = get_example_evaluation_dict(all_data_dfs, args,
                                                      proposed_equations,
                                                      num_variables)
save_example_evaluation_dict(args, example_evaluation_dict, logger)


# Create error table

In [ ]:
num_variables = 1
df_error = create_error_table(args, num_variables, proposed_equations, metric='error')
df_error

In [ ]:
indices_best_equations = list(df_error.index)

In [ ]:
from src.analyse_equations.utils import mean_std_in_error
num_variables = 1
df_error = create_error_table(args, num_variables, proposed_equations, metric='error_mse')
df_error

In [ ]:
num_variables = 1
df_error = create_error_table(args, num_variables, proposed_equations, metric='err_rel')
df_error

# Create constant table

In [ ]:
index = indices_best_equations[0]
create_constant_table( all_data_dfs,
        args,
        df_error.loc[index].loc['equation'],
        proposed_equations)

# Create heat map local

In [ ]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
indices = indices_best_equations[: 4] + [5,6]
metric = 'err_rel'
from definitions import dict_pre_to_infix


def save_system_error_heatmap(args, pd_dict):
    pd_constants_values = pd.DataFrame(pd_dict)
    pd_constants_values = pd_constants_values.rename(columns=dict_pre_to_infix)
    fig, ax = plt.subplots(figsize=(8, 6))
    mask = np.zeros(pd_constants_values.shape)
    mask[-8:, :] = True
    sns.heatmap(pd_constants_values, mask=mask, cmap='Oranges', linewidths=1.5,
                ax=ax, cbar=False)
    sns.heatmap(pd_constants_values, alpha=0.0, fmt=".2e", cmap='Oranges',
                cbar=False, annot=True, mask=mask)
    sns.heatmap(pd_constants_values, mask=np.logical_not(mask),
                cmap='BrBG', linewidths=1.5, ax=ax, cbar=False)
    sns.heatmap(pd_constants_values, alpha=0.0, fmt=".2", cmap='BrBG',
                cbar=False, annot=True, mask=np.logical_not(mask))
    ax.xaxis.tick_top()
    ax.set_xticklabels(rotation=45, labels=[label.get_text() for label in ax.get_xticklabels()],
                       ha='left')
    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)

    fig.tight_layout()
    save_path = args.save_path / 'error_per_system_heatmap.pdf'
    print(f"Heatmap saved @ {save_path}")
    fig.savefig(save_path)
    fig.show()



pd_dict = heatmap_error_per_system(
        all_data_dfs,
        args,
        df_error,
        proposed_equations,
        indices,
        metric
    )
pd.DataFrame(pd_dict)
save_system_error_heatmap(args, pd_dict)

### Create constant table

In [ ]:
index =  indices_best_equations[0]
pd_constants = create_constant_table(
    all_data_dfs,
    args,
    df_error.loc[index].loc['equation'],
    proposed_equations
)
save_constant_table(args, logger, pd_constants)
pd_constants

## Print units

In [ ]:
index =  indices_best_equations[0]
print_units_of_one_equation(args, df_error, index, proposed_equations)

In [ ]:
# Plot prediction

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from src.analyse_equations.plot_predictions_of_one_equation import get_short_and_sorted_df


def plot_prediction(args, filtered_dfs_test, filtered_dfs_train, tree, equation_id=''):
    filtered_dfs_test = get_short_ad_sorted_df(args, filtered_dfs_test)
    filtered_dfs_train= get_short_and_sorted_df(args, filtered_dfs_train)

    equation_infix = tree.rearrange_equation_infix_notation()[1]
    y_pred_train = tree.evaluate_subtree(-1, filtered_dfs_train)
    y_pred_test = tree.evaluate_subtree(-1, filtered_dfs_test)
    y_true_train = filtered_dfs_train['y'].to_numpy()
    y_true_test = filtered_dfs_test['y'].to_numpy()
    fig, (ax1, ax2) = plt.subplots(figsize=(10, 10), nrows=2, sharex=True, sharey=True)
    fig.suptitle(equation_infix)
    ax1.ticklabel_format(axis= 'y', style='sci', scilimits=(0,0))
    ax1.set_title('Train Data')
    ax1.scatter(range(len(y_true_train)), y_true_train, label='true', s=1)
    ax1.scatter(range(len(y_pred_train)), y_pred_train, label='prediction', s=1)
    ax1.legend(loc='upper right')
    ax1.set_ylabel('Friction Force')

    for index in filtered_dfs_train.drop_duplicates(subset='Video ID').index:
        ax1.axvline(x=index, ymax=1, color='black', linewidth=1)
        string = (#f"{int(filtered_dfs_train.loc[index, 'Video ID'])} "
                  f"{round(np.rad2deg(filtered_dfs_train.loc[index, 'tilt_angle']))}° "
                  f"{filtered_dfs_train.loc[index, 'excel_name']}")
        ax1.text(x=index,
                 y=0.00034,
                 s=string[:15],
                 rotation=90
                 )
    ax2.set_title('Test Data')
    ax2.scatter(range(len(y_true_test)), y_true_test, label='true', s=1)
    ax2.scatter(range(len(y_pred_test)), y_pred_test, label='prediction', s=1)
    ax2.legend(loc='upper right')
    ax2.set_ylabel('Friction Force')
    ax2.set_xlabel('Index in concatenated dataset')
    for index in filtered_dfs_test.drop_duplicates(subset='Video ID').index:
        ax2.axvline(x=index, ymax=1, color='black', linewidth=1)
        string = (#f"{int(filtered_dfs_test.loc[index, 'Video ID'])} "
                  f"{round(np.rad2deg(filtered_dfs_test.loc[index, 'tilt_angle']))}° "
                  f"{filtered_dfs_test.loc[index, 'excel_name']}")
        ax2.text(x=index,
                 y=0.00034,
                 s=string[:15],
                 rotation=90
                 )
    fig.tight_layout()
    save_path = args.save_path / f"equations/predictions/{equation_id}__{equation_infix.replace('/', ':')}.pdf"
    print(f"Saving prediction plot @: {save_path}")
    Path(save_path).parent.mkdir(exist_ok=True, parents=True)
    fig.savefig(save_path)
    plt.show()

for index in indices_best_equations[:70]:
    equation = proposed_equations[df_error.loc[index].loc['equation']]
    tree = map_equation_to_syntax_tree(args, df_error.loc[index].loc['equation'], infix=False, catch_exceptions=False)
    tree.constants_in_tree = equation['all_data']['train']['constants']
    plot_prediction(args, filtered_dfs_test, filtered_dfs_train, tree, equation_id=str(index))

## Error per system

In [ ]:
# index = 174
# plot_error_per_system(args, df_error, index, proposed_equations)

In [ ]:



########################################
############ histogram #############
########################################
# args.features = ['drop_length', 'y_center','avg_vel', 'width','adv', 'rec']
#
#
# index = 174
# equation = proposed_equations[df_error.loc[index].loc['equation']]
# tree = map_equation_to_syntax_tree(args, df_error.loc[index].loc['equation'], infix=False, catch_exceptions=False)
# tree.constants_in_tree = equation['all_data']['train']['constants']
# histogram_for_features(args, all_data_dfs, tree)

In [ ]:
########################################
###### difference between two eq #######
########################################
# index_0 = 174
# index_1 = 68
# abs_difference_between_equation(args, proposed_equations, df_error, all_data_dfs, index_0, index_1)
